# Modeling the MTA Hourly Ridership Dataset

This notebook builds and compares multiple regression models for forecasting hourly subway ridership using the processed outputs from the preprocessing notebook.

## 1. Imports and Load Preprocessed Inputs

Loads the processed training and testing feature matrices, target variables, and saved preprocessing artifacts from the previous notebook.

In [8]:
import pandas as pd
import numpy as np
import joblib

from scipy.sparse import load_npz

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
X_train_processed = load_npz("../CSVs/X_train_processed.npz")
X_test_processed = load_npz("../CSVs/X_test_processed.npz")

y_train = pd.read_parquet("../CSVs/y_train.parquet")["ridership_total"]
y_test = pd.read_parquet("../CSVs/y_test.parquet")["ridership_total"]

all_feature_names = np.load("../CSVs/all_feature_names.npy", allow_pickle=True)
preprocessor = joblib.load("../CSVs/preprocessor.joblib")

## 2. Confirm the Loaded Data Shapes

Checks that the processed matrices and target vectors were loaded correctly before modeling begins.

In [10]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Number of transformed features:", len(all_feature_names))

X_train_processed shape: (6034307, 485)
X_test_processed shape: (1508577, 485)
y_train shape: (6034307,)
y_test shape: (1508577,)
Number of transformed features: 485


## 3. Define the Modeling Objective

Specifies the target, problem type, and project framing for the modeling stage.

In [11]:
target_name = "ridership_total"

print("Target:", target_name)

Target: ridership_total


## 4. Create a Chronological Validation Split from the Training Set

Because this project is a forecasting problem, model evaluation should preserve the time order of the data. A random validation split would mix earlier and later observations and could give an overly optimistic estimate of model performance. To avoid this, the last portion of the training data is used as a validation set. This allows model selection to reflect how the models would perform when predicting future observations from past data.

In [12]:
val_split_idx = int(len(y_train) * 0.8)

X_fit = X_train_processed[:val_split_idx]
X_val = X_train_processed[val_split_idx:]

y_fit = y_train.iloc[:val_split_idx].copy()
y_val = y_train.iloc[val_split_idx:].copy()

print("X_fit shape:", X_fit.shape)
print("X_val shape:", X_val.shape)
print("y_fit shape:", y_fit.shape)
print("y_val shape:", y_val.shape)

X_fit shape: (4827445, 485)
X_val shape: (1206862, 485)
y_fit shape: (4827445,)
y_val shape: (1206862,)


## 5. Define Evaluation Metrics

This project predicts `ridership_total`, which is a continuous response variable, so regression metrics are more appropriate than classification metrics such as accuracy or F1. I use MAE, RMSE, WMAPE, and R² to compare models from multiple perspectives. MAE shows average absolute error in ridership units, RMSE places more weight on larger errors, WMAPE scales the error relative to total demand, and R² provides a general measure of explained variation.

In [13]:
def wmape(y_true, y_pred):
    numerator = np.abs(y_true - y_pred).sum()
    denominator = np.abs(y_true).sum()
    return np.nan if denominator == 0 else numerator / denominator


def evaluate_regression(y_true, y_pred, model_name, split_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return {
        "Model": model_name,
        "Split": split_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
        "WMAPE": wmape(y_true, y_pred)
    }

## 6. Initialize Storage for Candidate Models and Results

We create our containers to store fitted models and their validation performance.

In [14]:
candidate_models = {}
results = []

## 7. Baseline Model: Dummy Regressor

I begin with a baseline model using `DummyRegressor` so that the more advanced models can be evaluated against a simple benchmark. This model predicts the mean ridership from the fitting data and does not use feature information. If the more advanced models do not improve meaningfully over this baseline, then they are not adding much predictive value.

In [15]:
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_fit, y_fit)

dummy_val_pred = dummy_model.predict(X_val)

dummy_result = evaluate_regression(
    y_true=y_val,
    y_pred=dummy_val_pred,
    model_name="DummyRegressor_mean",
    split_name="validation"
)

candidate_models["DummyRegressor_mean"] = dummy_model
results.append(dummy_result)

pd.DataFrame([dummy_result])

,Model,Split,MAE,RMSE,R2,WMAPE
0,DummyRegressor_mean,validation,324.678443,685.735215,-0.001584,1.016972


## 8. Model 1: Ridge Regression

`Ridge` was selected as the first main candidate model because the processed feature matrix is large, high-dimensional, and sparse due to one-hot encoding. Ridge regression is well suited to this type of data because it handles many correlated predictors and applies L2 regularization to reduce overfitting. The main hyperparameter for Ridge is `alpha`, which controls the strength of regularization. I test several values across a broad range to evaluate how much shrinkage improves generalization.

In [16]:
ridge_alphas = [0.1, 1.0, 10.0, 50.0, 100.0]

for alpha in ridge_alphas:
    model_name = f"Ridge_alpha={alpha}"
    
    ridge_model = Ridge(alpha=alpha, solver="lsqr")
    ridge_model.fit(X_fit, y_fit)
    
    ridge_val_pred = ridge_model.predict(X_val)
    
    ridge_result = evaluate_regression(
        y_true=y_val,
        y_pred=ridge_val_pred,
        model_name=model_name,
        split_name="validation"
    )
    
    candidate_models[model_name] = ridge_model
    results.append(ridge_result)

ridge_results = pd.DataFrame(results)
ridge_results[ridge_results["Model"].str.contains("Ridge")].sort_values("MAE")

,Model,Split,MAE,RMSE,R2,WMAPE
5,Ridge_alpha=100.0,validation,91.319550,223.418333,0.893681,0.286035
4,Ridge_alpha=50.0,validation,91.327268,223.417662,0.893681,0.286059
3,Ridge_alpha=10.0,validation,91.333519,223.417146,0.893682,0.286079
2,Ridge_alpha=1.0,validation,91.334936,223.417032,0.893682,0.286083
1,Ridge_alpha=0.1,validation,91.335078,223.417021,0.893682,0.286084


## 9. Model 2: SGD Regressor

`SGDRegressor` was selected as a second sparse-compatible linear model because it is designed to scale efficiently to very large datasets. Unlike Ridge, which solves a regularized linear system directly, SGD learns iteratively and can be more computationally practical for large-scale problems. I tune `alpha` and `penalty` because these hyperparameters strongly affect regularization strength and how the model controls complexity. Testing both `l2` and `elasticnet` provides a comparison between pure shrinkage and a mixed regularization approach.

In [17]:
sgd_param_grid = [
    {"alpha": 1e-4, "penalty": "l2"},
    {"alpha": 1e-3, "penalty": "l2"},
    {"alpha": 1e-4, "penalty": "elasticnet"},
    {"alpha": 1e-3, "penalty": "elasticnet"},
]

for params in sgd_param_grid:
    model_name = f"SGDRegressor_alpha={params['alpha']}_penalty={params['penalty']}"
    
    sgd_model = SGDRegressor(
        loss="squared_error",
        penalty=params["penalty"],
        alpha=params["alpha"],
        max_iter=2000,
        tol=1e-3,
        random_state=42
    )
    
    sgd_model.fit(X_fit, y_fit)
    
    sgd_val_pred = sgd_model.predict(X_val)
    
    sgd_result = evaluate_regression(
        y_true=y_val,
        y_pred=sgd_val_pred,
        model_name=model_name,
        split_name="validation"
    )
    
    candidate_models[model_name] = sgd_model
    results.append(sgd_result)

sgd_results = pd.DataFrame(results)
sgd_results[sgd_results["Model"].str.contains("SGDRegressor")].sort_values("MAE")

,Model,Split,MAE,RMSE,R2,WMAPE
9,SGDRegressor_alpha=0.001_penalty=elasticnet,validation,92.448389,224.409619,0.892735,0.289571
7,SGDRegressor_alpha=0.001_penalty=l2,validation,92.483208,224.432731,0.892713,0.289680
8,SGDRegressor_alpha=0.0001_penalty=elasticnet,validation,92.827004,224.303471,0.892837,0.290757
6,SGDRegressor_alpha=0.0001_penalty=l2,validation,92.846802,224.307459,0.892833,0.290819


## 10. Compare Validation Results Across Models

After fitting the candidate models, I compare their validation performance using the selected regression metrics. The goal is not only to identify the model with the strongest numerical performance, but also to evaluate whether that performance is meaningfully better than the baseline and whether the model remains computationally practical for this dataset.

In [18]:
results_df = pd.DataFrame(results).sort_values(
    by=["MAE", "RMSE", "WMAPE"],
    ascending=[True, True, True]
).reset_index(drop=True)

results_df

,Model,Split,MAE,RMSE,R2,WMAPE
0,Ridge_alpha=100.0,validation,91.319550,223.418333,0.893681,0.286035
1,Ridge_alpha=50.0,validation,91.327268,223.417662,0.893681,0.286059
2,Ridge_alpha=10.0,validation,91.333519,223.417146,0.893682,0.286079
3,Ridge_alpha=1.0,validation,91.334936,223.417032,0.893682,0.286083
4,Ridge_alpha=0.1,validation,91.335078,223.417021,0.893682,0.286084
5,SGDRegressor_alpha=0.001_penalty=elasticnet,validation,92.448389,224.409619,0.892735,0.289571
6,SGDRegressor_alpha=0.001_penalty=l2,validation,92.483208,224.432731,0.892713,0.289680
7,SGDRegressor_alpha=0.0001_penalty=elasticnet,validation,92.827004,224.303471,0.892837,0.290757
8,SGDRegressor_alpha=0.0001_penalty=l2,validation,92.846802,224.307459,0.892833,0.290819
9,DummyRegressor_mean,validation,324.678443,685.735215,-0.001584,1.016972


## 11. Select the Best Validation Model

The best model is selected based primarily on validation performance, with emphasis on MAE, RMSE, and WMAPE. Since this is a forecasting problem with large-scale data, model choice should also account for scalability and stability rather than focusing only on raw performance.

In [19]:
best_model_name = results_df.iloc[0]["Model"]
best_model_template = candidate_models[best_model_name]

print("Best validation model:", best_model_name)
results_df.head(10)

Best validation model: Ridge_alpha=100.0


,Model,Split,MAE,RMSE,R2,WMAPE
0,Ridge_alpha=100.0,validation,91.319550,223.418333,0.893681,0.286035
1,Ridge_alpha=50.0,validation,91.327268,223.417662,0.893681,0.286059
2,Ridge_alpha=10.0,validation,91.333519,223.417146,0.893682,0.286079
3,Ridge_alpha=1.0,validation,91.334936,223.417032,0.893682,0.286083
4,Ridge_alpha=0.1,validation,91.335078,223.417021,0.893682,0.286084
5,SGDRegressor_alpha=0.001_penalty=elasticnet,validation,92.448389,224.409619,0.892735,0.289571
6,SGDRegressor_alpha=0.001_penalty=l2,validation,92.483208,224.432731,0.892713,0.289680
7,SGDRegressor_alpha=0.0001_penalty=elasticnet,validation,92.827004,224.303471,0.892837,0.290757
8,SGDRegressor_alpha=0.0001_penalty=l2,validation,92.846802,224.307459,0.892833,0.290819
9,DummyRegressor_mean,validation,324.678443,685.735215,-0.001584,1.016972


## 12. Second run Check: Compare Ridge With and Without the Year Feature

This comparison tests whether the `year` feature contributes meaningful predictive signal. Based on mentor feedback, `month` is more likely to capture recurring seasonal effects, while `year` may provide limited value because the dataset spans a relatively short time period. Comparing the same Ridge model with and without `year` helps determine whether it should remain in the final feature set.

In [20]:
feature_names_series = pd.Series(all_feature_names)

print(feature_names_series[feature_names_series == "year"])

477    year
dtype: object


In [21]:
if "year" in feature_names_series.values:
    keep_mask = feature_names_series != "year"
    
    X_fit_no_year = X_fit[:, keep_mask.values]
    X_val_no_year = X_val[:, keep_mask.values]
    
    ridge_with_year = Ridge(alpha=1.0, solver="lsqr")
    ridge_with_year.fit(X_fit, y_fit)
    pred_with_year = ridge_with_year.predict(X_val)
    
    ridge_without_year = Ridge(alpha=1.0, solver="lsqr")
    ridge_without_year.fit(X_fit_no_year, y_fit)
    pred_without_year = ridge_without_year.predict(X_val_no_year)
    
    year_comparison = pd.DataFrame([
        evaluate_regression(y_val, pred_with_year, "Ridge_with_year", "validation"),
        evaluate_regression(y_val, pred_without_year, "Ridge_without_year", "validation")
    ])
    
    display(year_comparison.sort_values("MAE"))
else:
    print("The transformed feature list does not contain a standalone 'year' column.")

,Model,Split,MAE,RMSE,R2,WMAPE
0,Ridge_with_year,validation,91.334936,223.417032,0.893682,0.286083
1,Ridge_without_year,validation,91.336769,223.416982,0.893682,0.286089


## 13. Refit the Best Model on the Full Training Set

This retrains the selected model using the full training dataset before final testing.

In [22]:
final_model = clone(best_model_template)
final_model.fit(X_train_processed, y_train)

Ridge(alpha=100.0, solver='lsqr')

## 14. Evaluate the Final Model on the Held-Out Test Set

After selecting the strongest validation model, I refit it on the full training dataset and evaluate it once on the held-out test set. This produces the final estimate of how the selected model performs on unseen future data.

In [24]:
y_test_pred = final_model.predict(X_test_processed)

test_result = evaluate_regression(
    y_true=y_test,
    y_pred=y_test_pred,
    model_name=best_model_name,
    split_name="test"
)

test_results_df = pd.DataFrame([test_result])
test_results_df

,Model,Split,MAE,RMSE,R2,WMAPE
0,Ridge_alpha=100.0,test,73.546418,170.95767,0.936299,0.233051


## 15. Modeling Summary

This notebook served as the modeling point in the pipeline. I built and compared three regression models to forecast our target: hourly subway ridership. The models chosen were a baseline `DummyRegressor`, a `Ridge` model, and an `SGDRegressor`. These models were chosen because `ridership_total` is continuous, and the feature matrix that resulted from preprocessing was large and sparse after one-hot encoding. These models are sparse-friendly and made the most sense to start with.

As we're forecasting, I used a chronological validation split instead of a random one so the model is trained chronologically. MAE, RMSE, WMAPE, and R² as our metrics because these are more appropriate for the continuous prediction. I also did a bit of light hyperparameter tuning with multiple `alpha` values for Ridge and multiple `alpha` and `penalty` combinations for `SGDRegressor`.

The final model was chosen mostly from validation performance, but I also wanted to keep practicality in mind. I was not looking for the strongest score alone if the model was going to be harder to work with, harder to scale, or less stable overall. I also checked whether removing the `year` feature made any difference, since `month` seemed like the more realistic seasonal feature given the shorter time span of the dataset.

The main question during this modeling stage was whether hourly `ridership_total` could actually be forecasted well using the station, calendar, and lag-based features that were engineered earlier in the project. Based on the model comparisons, the answer is yes. The strongest model in this notebook ended up being `Ridge(alpha=100.0)`, and it performed better than both the baseline `DummyRegressor` and the different `SGDRegressor` setups that were tested.

On the held-out test set, the final Ridge model produced an MAE of about 73.55, an RMSE of about 170.96, an R² of about 0.9363, and a WMAPE of about 0.2331. Overall, that means the model was able to explain a large amount of the variation in hourly ridership and gave much stronger forecasts than the baseline model.

The feature check also showed that `year` did not really make a meaningful difference in validation performance, which suggests it was not adding much useful signal over the time period covered by this dataset. `Month` ended up being the more useful calendar feature since it works better as a seasonal indicator. Overall, the results from this notebook show that hourly subway ridership can be forecasted effectively in this project using a sparse-compatible regularized linear model.